# Silver - ecommerce_enderecos

Desenvolvido por: Ygor Moraes

Este notebook lê a Bronze `ecommerce_enderecos` e grava a Silver tratada em Delta.

Regras aplicadas:
- deduplicar endereços por `id_endereco`;
- converter `id_endereco` e `id_cliente` para integer;
- limpar campos textuais com remoção de espaços extras;
- padronizar `estado` como UF em letras maiúsculas;
- padronizar `cep`, mantendo apenas valores com 8 dígitos e convertendo CEPs inválidos para nulo;
- converter `latitude` e `longitude` para double;
- converter `is_principal` para boolean;
- adicionar `silver_processed_at`;
- manter particionamento por `ano` e `mes`.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Silver de endereços.

from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    upper,
    regexp_replace,
    current_timestamp,
    row_number
)

from pyspark.sql.window import Window

BRONZE_TABLE = "ecommerce_enderecos"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

SILVER_TABLE = "ecommerce_enderecos"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

KEY_COLUMNS = ["id_endereco"]

SILVER_WRITE_MODE = "overwrite"

UF_VALIDAS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
]

BRONZE_REQUIRED_COLUMNS = [
    "id_endereco",
    "id_cliente",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "cidade",
    "estado",
    "latitude",
    "longitude",
    "apelido",
    "is_principal",
    "bronze_ingested_at",
    "bronze_source_file",
    "bronze_record_hash",
    "ano",
    "mes"
]

SILVER_REQUIRED_COLUMNS = [
    "id_endereco",
    "id_cliente",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "cidade",
    "estado",
    "latitude",
    "longitude",
    "apelido",
    "is_principal",
    "bronze_ingested_at",
    "bronze_source_file",
    "silver_processed_at",
    "ano",
    "mes"
]

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Origem Bronze: {BRONZE_PATH}")
print(f"Destino Silver: {SILVER_PATH}")
print(f"Modo de escrita: {SILVER_WRITE_MODE}")

In [0]:
# Lê a Bronze e valida se as colunas necessárias existem.

df_bronze = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

bronze_columns = df_bronze.columns

missing_bronze_columns = [
    c for c in BRONZE_REQUIRED_COLUMNS
    if c not in bronze_columns
]

if missing_bronze_columns:
    raise Exception(f"Colunas obrigatórias ausentes na Bronze: {missing_bronze_columns}")

total_bronze = df_bronze.count()

print("Bronze lida com sucesso.")
print(f"Total de registros na Bronze: {total_bronze}")

df_bronze.printSchema()

display(df_bronze.limit(10))

In [0]:
# Valida chave e campos críticos antes das transformações.

total_registros = df_bronze.count()

ids_nulos = df_bronze.filter(col("id_endereco").isNull()).count()

ids_duplicados = (
    df_bronze
    .groupBy("id_endereco")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Total de registros: {total_registros}")
print(f"IDs nulos: {ids_nulos}")
print(f"IDs duplicados: {ids_duplicados}")

display(
    df_bronze
    .select("estado", "cep", "latitude", "longitude", "is_principal")
    .limit(10)
)

In [0]:
# Aplica limpeza, padronização e tipagem dos campos da Silver.

df_silver_base = (
    df_bronze
    .select(
        col("id_endereco").cast("int").alias("id_endereco"),
        col("id_cliente").cast("int").alias("id_cliente"),

        trim(col("logradouro")).alias("logradouro"),
        trim(col("numero")).alias("numero"),
        trim(col("complemento")).alias("complemento"),
        trim(col("bairro")).alias("bairro"),

        when(
            regexp_replace(trim(col("cep")), "[^0-9]", "").rlike("^[0-9]{8}$"),
            regexp_replace(trim(col("cep")), "[^0-9]", "")
        ).otherwise(None).alias("cep"),

        trim(col("cidade")).alias("cidade"),
        upper(trim(col("estado"))).alias("estado"),

        regexp_replace(trim(col("latitude")), ",", ".").cast("double").alias("latitude"),
        regexp_replace(trim(col("longitude")), ",", ".").cast("double").alias("longitude"),

        trim(col("apelido")).alias("apelido"),

        when(upper(trim(col("is_principal"))).isin("TRUE", "1", "SIM"), True)
        .when(upper(trim(col("is_principal"))).isin("FALSE", "0", "NAO", "NÃO"), False)
        .otherwise(None)
        .alias("is_principal"),

        col("bronze_ingested_at"),
        col("bronze_source_file"),
        current_timestamp().alias("silver_processed_at"),
        col("ano"),
        col("mes")
    )
)

display(df_silver_base.limit(10))

In [0]:
# Mantém apenas registros com chave, UF, coordenadas e flag principal válidas.
# CEP nulo não descarta o endereço, pois o registro ainda pode ser útil nas análises.

df_silver_validado = (
    df_silver_base
    .filter(col("id_endereco").isNotNull())
    .filter(col("id_cliente").isNotNull())
    .filter(col("estado").isin(UF_VALIDAS))
    .filter(col("latitude").between(-34.0, 6.0))
    .filter(col("longitude").between(-74.0, -34.0))
    .filter(col("is_principal").isNotNull())
)

total_validado = df_silver_validado.count()

ceps_nulos = df_silver_validado.filter(col("cep").isNull()).count()

ceps_invalidos = (
    df_silver_validado
    .filter(col("cep").isNotNull())
    .filter(~col("cep").rlike("^[0-9]{8}$"))
    .count()
)

print(f"Total antes das regras: {total_registros}")
print(f"Total após regras críticas: {total_validado}")
print(f"Registros removidos: {total_registros - total_validado}")
print(f"CEPs nulos na Silver validada: {ceps_nulos}")
print(f"CEPs inválidos após tratamento: {ceps_invalidos}")

if ceps_invalidos > 0:
    raise Exception("Ainda existem CEPs inválidos na Silver validada.")

In [0]:
# Deduplica endereços pela chave principal.

window_enderecos = (
    Window
    .partitionBy("id_endereco")
    .orderBy(col("bronze_ingested_at").desc())
)

df_silver = (
    df_silver_validado
    .withColumn("row_number", row_number().over(window_enderecos))
    .filter(col("row_number") == 1)
    .drop("row_number")
)

total_silver = df_silver.count()

duplicados_silver = (
    df_silver
    .groupBy("id_endereco")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Total Silver final: {total_silver}")
print(f"IDs duplicados na Silver: {duplicados_silver}")

display(df_silver.limit(10))

In [0]:
# Valida se a Silver possui as colunas esperadas.

silver_columns = df_silver.columns

missing_silver_columns = [
    c for c in SILVER_REQUIRED_COLUMNS
    if c not in silver_columns
]

if missing_silver_columns:
    raise Exception(f"Colunas obrigatórias ausentes na Silver: {missing_silver_columns}")

print("Schema final da Silver validado.")

df_silver.printSchema()

In [0]:
# Grava a Silver de endereços em Delta.

(
    df_silver
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_PATH)
)

print("Silver gravada com sucesso.")
print(f"Path: {SILVER_PATH}")

In [0]:
# Lê a Silver gravada e valida volume e chave.

df_silver_gravada = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

total_gravado = df_silver_gravada.count()

duplicados_gravados = (
    df_silver_gravada
    .groupBy("id_endereco")
    .count()
    .filter(col("count") > 1)
    .count()
)

ceps_nulos_gravados = df_silver_gravada.filter(col("cep").isNull()).count()

ceps_invalidos_gravados = (
    df_silver_gravada
    .filter(col("cep").isNotNull())
    .filter(~col("cep").rlike("^[0-9]{8}$"))
    .count()
)

print(f"Total Silver gravada: {total_gravado}")
print(f"IDs duplicados na Silver gravada: {duplicados_gravados}")
print(f"CEPs nulos mantidos: {ceps_nulos_gravados}")

display(df_silver_gravada.limit(10))